# BrieFYI 학습 데이터 생성 파이프라인 — Colab 실행용 (`make_train_data`)

로컬 환경에서 공유 DB에 직접 접속할 수 없을 때, 이 노트북으로 Colab에서 바로
`make_train_data` 파이프라인을 실행할 수 있습니다. DB에서 뉴스 기사를 읽어와
군집화하고, 군집별 JSON 파일을 생성합니다 — 이 파일들은 이후 사람이 직접
Claude 채팅에 넣어 학습 데이터를 만드는 데 씁니다.

**GPU가 필요 없습니다.** 임베딩은 로컬 모델이 아니라 Hugging Face Inference API를
통해 호출하므로, 런타임 유형은 CPU로 두고 실행해도 됩니다.

필요한 것은 DB 접속 정보(`DATABASE_URL`)와 Hugging Face 토큰(`HF_TOKEN`)뿐입니다.

## 1. 번들 업로드

로컬 PC(레포 루트)에서 아래 경로들을 **상대 경로 구조를 그대로 유지한 채** zip으로
묶어 업로드하세요 (예: `make_train_data_bundle.zip`):

- `finetune/make_train_data/` (파이프라인 코드 — `finetune/` 하위 경로를 그대로 유지해야 합니다)
- `db/` (레포 루트, DB 연결·쿼리 유틸)
- `config.py` (레포 루트)
- `rag_latest/` (레포 루트, 임베딩 호출에 필요)
- `requirements.txt` (레포 루트, 참고용 — 실제 설치는 4번 셀의 최소 의존성만 사용합니다)

레포 루트에서 예:
```bash
zip -r make_train_data_bundle.zip finetune/make_train_data db config.py rag_latest requirements.txt
```

`finetune/src/summarize_ft/`(학습 노트북 전용 코드)는 이 번들에 포함하지 않습니다.

> `make_train_data` 코드는 레포 루트를 상위 디렉터리 두 단계로 찾는 방식
> (`Path(__file__).resolve().parents[2]`)에 의존합니다. `make_train_data/` 폴더를
> `finetune/` 없이 최상위로 옮기면 동작하지 않으니, 폴더 구조를 반드시 그대로 유지하세요.

In [ ]:
from google.colab import files
import zipfile, os

print("make_train_data_bundle.zip을 선택하세요 (finetune/make_train_data/, db/, config.py, rag_latest/, requirements.txt 포함)")
uploaded = files.upload()
bundle_name = list(uploaded.keys())[0]
with zipfile.ZipFile(bundle_name, "r") as z:
    z.extractall("/content")
os.chdir("/content")

## 2. `.env` 구성

파이프라인이 필요로 하는 값은 두 가지입니다:

- `DATABASE_URL` — 공유 DB 접속 문자열
- `HF_TOKEN` — 임베딩 호출에 쓰는 Hugging Face 토큰

아래 셀을 실행하면 값을 입력하라는 프롬프트가 뜹니다(화면에 노출되지 않도록 `getpass` 사용).
입력한 값은 `/content/.env`에 저장되고, `make_train_data`의 설정 모듈이 이 파일을 읽습니다.

In [ ]:
import os
from getpass import getpass

database_url = os.environ.get("DATABASE_URL") or getpass("DATABASE_URL: ")
hf_token = os.environ.get("HF_TOKEN") or getpass("HF_TOKEN: ")
with open(".env", "w") as f:
    f.write(f"DATABASE_URL={database_url}\nHF_TOKEN={hf_token}\n")

## 3. 의존성 설치

`make_train_data` 파이프라인은 DB 조회와 HTTP 호출(임베딩 API)만 하므로, `torch`나
`transformers` 같은 무거운 패키지는 필요 없습니다. 아래 최소 의존성만 설치합니다.

In [ ]:
!pip install -q psycopg[binary] python-dotenv requests

## 4. 파이프라인 실행

기사 조회 → 군집화 → 엔티티/임베딩 기반 정제 → onefact 후보 선정 → 군집별 JSON export까지
전체 파이프라인을 한 번에 실행합니다.

- `--since`는 선택 옵션입니다 — 특정 날짜 이후 기사만 대상으로 하고 싶을 때 씁니다. 아래
  날짜(`2026-08-01`)는 예시이니 필요에 맞게 바꾸세요. 생략하면 전체 기간을 대상으로 합니다.
- `make_train_data` 패키지는 `finetune/` 아래 있고, `python -m make_train_data.cli`로
  실행하려면 `finetune/`이 현재 디렉터리여야 하므로 아래 셀은 그 디렉터리로 이동한 뒤
  실행합니다(노트북 자체의 현재 디렉터리는 `/content`로 유지됩니다).
- 결과는 `/content/finetune/output/`에 군집별 JSON 파일로 쌓입니다.

In [ ]:
!cd /content/finetune && python -m make_train_data.cli run --out-dir output --since 2026-08-01

## 5. 결과 다운로드

생성된 `output/` 디렉터리를 zip으로 묶어 로컬로 내려받습니다. 압축이 끝나면 브라우저에서
다운로드 창이 뜹니다.

In [ ]:
from google.colab import files
import shutil

shutil.make_archive("make_train_data_output", "zip", "/content/finetune/output")
files.download("make_train_data_output.zip")